# 03 - Task-aware structured reward

**학습 목표**: GLM-OCR Stage-4 RL의 NED, field F1, JSON validity, 반복 penalty를 작은 예제로 결합합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 `json` 등 Python 표준 라이브러리만 사용합니다.

실제 GRPO rollout/reward scale은 재현하지 않습니다.

In [ ]:
import json
# 동적 계획법 표를 한 행씩 갱신해 전체 2차원 표보다 메모리를 적게 사용합니다.

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def text_reward(pred, truth):
    ned = edit_distance(pred, truth) / max(1, len(pred), len(truth))
    repetition = max(0, pred.count('TOTAL') - 1)
    return 1 - ned - 0.1 * repetition

def kie_reward(pred_json, truth):
    try:
        pred = json.loads(pred_json)
        valid = 1.0
    except json.JSONDecodeError:
        return -1.0
    pred_items, truth_items = set(pred.items()), set(truth.items())
    tp = len(pred_items & truth_items)
    precision = tp / max(1, len(pred_items))
    recall = tp / max(1, len(truth_items))
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return f1 + 0.1 * valid

print('clean text:', text_reward('TOTAL 42', 'TOTAL 42'))
print('repeated text:', text_reward('TOTAL TOTAL 42', 'TOTAL 42'))
truth = {'date': '2026-09-05', 'total': '42'}
print('valid KIE:', kie_reward('{"date": "2026-09-05", "total": "42"}', truth))
print('malformed KIE:', kie_reward('{date: 2026}', truth))
assert text_reward('TOTAL 42', 'TOTAL 42') > text_reward('TOTAL TOTAL 42', 'TOTAL 42')
assert kie_reward('{bad json}', truth) == -1.0


Text, formula, table, KIE는 오류 구조가 다르므로 하나의 문자열 점수만 쓰면 안 됩니다. 논문은 각각 NED, CDM, TEDS, field F1에 repetition, tag closure, JSON parsing 등의 제약을 더합니다.